# Conventional Beamforming
## Tutorial 4: Delay-and-Sum Beamforming, Beam Patterns, and Limitations

Conventional beamforming (CBF), also called **Delay-and-Sum** (D&S), is the simplest DOA estimation method.  Topics:

1. **Principle** – matched filter in space
2. **Beam pattern** – sidelobes, mainlobe width
3. **Beamwidth and angular resolution**
4. **Limitations** – sidelobe masking, poor resolution
5. **Window functions** for sidelobe control

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.classical import ConventionalBeamforming

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array = UniformLinearArray(M=M, d=0.5)
signal_model = SignalModel(array)
cbf = ConventionalBeamforming(array)

print("Setup complete.  M =", M)

## 1. Principle of Conventional Beamforming

The CBF **steers** the array toward a candidate angle $\theta$ by applying conjugate phase shifts (matched filter):

$$\mathbf{w}(\theta) = \frac{1}{M}\mathbf{a}(\theta)$$

The output power (beam pattern) is then:

$$P_{\text{CBF}}(\theta) = \mathbf{w}^H(\theta)\hat{\mathbf{R}}\mathbf{w}(\theta) = \frac{1}{M^2}\mathbf{a}^H(\theta)\hat{\mathbf{R}}\mathbf{a}(\theta)$$

Peaks in $P_{\text{CBF}}$ indicate the likely DOA locations.

In [ ]:
doas_true = np.deg2rad([-20.0, 15.0])
snr_db = 10
N = 200
angle_grid = np.linspace(-np.pi/2, np.pi/2, 1801)

X, S, Noise = signal_model.generate_signals(
    doas=doas_true, N_snapshots=N, snr_db=snr_db, seed=42)

beam = cbf.beam_pattern(X, angle_grid)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(np.rad2deg(angle_grid), 10*np.log10(beam + 1e-10), 'b-', lw=2)
for th in doas_true:
    ax.axvline(np.rad2deg(th), color='r', ls='--',
               label=f'True {np.rad2deg(th):.0f}°')
ax.set_xlabel('θ (°)'); ax.set_ylabel('P_CBF (dB)')
ax.set_title(f'CBF Beam Pattern  (M={M}, SNR={snr_db} dB, N={N})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-40, 5)
plt.tight_layout(); plt.show()

# DOA estimation
doas_est = cbf.estimate(X, K=len(doas_true))
print(f"True DOAs  : {np.round(np.rad2deg(doas_true), 2)} °")
print(f"CBF estimates: {np.round(np.rad2deg(doas_est), 2)} °")

## 2. Beam Pattern Anatomy

A standard ULA beam pattern has:
- **Mainlobe** – centred on the steering direction, width $\approx 2/Md$ rad
- **First sidelobe** – $\approx 13$ dB below mainlobe (for uniform weighting)
- **Nulls** – periodic zeros
- **Grating lobes** – if $d > \lambda/2$

In [ ]:
# Single-source case: examine the beam pattern shape carefully
theta_steer = np.deg2rad(0)    # Broadside

X_single, _, _ = signal_model.generate_signals(
    doas=[theta_steer], N_snapshots=500, snr_db=20, seed=1)
beam_single = cbf.beam_pattern(X_single, angle_grid)
beam_dB = 10*np.log10(beam_single + 1e-10)

# Find mainlobe and first sidelobe
i_main = np.argmax(beam_dB)
# 3dB beamwidth
half_power = beam_dB[i_main] - 3
above = beam_dB >= half_power
# width
left = np.where(above[:i_main])[0]
right = np.where(above[i_main:])[0]
if len(left) and len(right):
    bw = np.rad2deg(angle_grid[i_main + right[-1]] - angle_grid[left[0]])
else:
    bw = np.nan

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(np.rad2deg(angle_grid), beam_dB, 'b-', lw=2)
ax.axhline(beam_dB[i_main] - 3, color='orange', ls='--',
           label=f'–3 dB  (BW ≈ {bw:.1f}°)')
ax.axhline(-13, color='purple', ls=':', label='–13 dB (sidelobe level)')
ax.set_xlabel('θ (°)'); ax.set_ylabel('Beam Pattern (dB)')
ax.set_title('CBF Beam Pattern — Single Source at 0°')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-40, 5)
plt.tight_layout(); plt.show()

print(f"3-dB beamwidth: {bw:.2f}°")
print(f"Theory: {np.rad2deg(2/(M*0.5)):.2f}°")

## 3. Resolution Limit

CBF cannot resolve two sources separated by less than the mainlobe beamwidth (Rayleigh limit):

$$\Delta\theta_{\min} \approx \frac{0.886\lambda}{L} = \frac{0.886}{(M-1)d}$$

Let's verify this with a controlled experiment.

In [ ]:
separations_deg = [15, 8, 5, 2]     # test angular separations

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
center = np.deg2rad(0)
rayleigh_bw = np.rad2deg(0.886 / ((M-1)*0.5))

for ax, sep_deg in zip(axes.flatten(), separations_deg):
    th1 = center - np.deg2rad(sep_deg/2)
    th2 = center + np.deg2rad(sep_deg/2)
    X2, _, _ = signal_model.generate_signals(
        doas=[th1, th2], N_snapshots=300, snr_db=15, seed=7)
    bp = cbf.beam_pattern(X2, angle_grid)
    ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp + 1e-10), lw=2)
    ax.axvline(np.rad2deg(th1), color='r', ls='--')
    ax.axvline(np.rad2deg(th2), color='r', ls='--')
    resolved = '✓ Resolved' if sep_deg > rayleigh_bw else '✗ Unresolved'
    ax.set_title(f'Separation = {sep_deg}°  [{resolved}]')
    ax.set_xlabel('θ (°)'); ax.set_ylabel('dB')
    ax.set_ylim(-40, 5); ax.grid(True, alpha=0.3)

plt.suptitle(f'CBF Resolution  (M={M}, d=0.5λ, Rayleigh BW ≈ {rayleigh_bw:.1f}°)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Window Functions for Sidelobe Control

Tapering the array weights (like tapering an FFT window) trades mainlobe width for reduced sidelobes.

Common windows:
- **Rectangular** (uniform): –13 dB sidelobe, narrow mainlobe
- **Hann**: –31 dB sidelobe, 1.5× wider mainlobe
- **Chebyshev**: equiripple, specified sidelobe level

In [ ]:
from numpy.fft import fft, fftshift

windows = {
    'Rectangular': np.ones(M),
    'Hann':        np.hanning(M),
    'Hamming':     np.hamming(M),
    'Blackman':    np.blackman(M),
}

# Single source at 0° (high SNR to see the pattern cleanly)
X_win, _, _ = signal_model.generate_signals(
    doas=[0.0], N_snapshots=1000, snr_db=30, seed=0)
R_win = X_win @ X_win.conj().T / 1000

fig, ax = plt.subplots(figsize=(13, 7))

for name, w in windows.items():
    W = w / np.sum(np.abs(w))           # normalise
    bp = np.zeros(len(angle_grid))
    for i, th in enumerate(angle_grid):
        a = array.steering_vector(th)
        bp[i] = np.real(W.conj() @ R_win @ W) if False else                 np.abs(W @ a)**2  # array factor only
    bp /= bp.max()
    ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp + 1e-12), lw=2, label=name)

ax.set_xlabel('θ (°)'); ax.set_ylabel('Normalised AF (dB)')
ax.set_title(f'Window Comparison  (M={M}, single source at 0°)')
ax.set_ylim(-60, 2); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

- CBF is simple, robust, always works — but has limited resolution (Rayleigh limit ∝ $1/L$).
- The –13 dB sidelobe level can mask weak sources near a strong one.
- Window functions reduce sidelobes but widen the mainlobe.
- For higher resolution without spatial smoothing cost, use subspace methods (Tutorial 7+).

## Exercises
1. Show that the CBF beam pattern equals the magnitude-squared of the spatial DFT of the array weights.
2. For $M=16, d=0.5\lambda$, find the minimum SNR (dB) needed to locate a single source at 20° with error < 1° using CBF.
3. Implement a **Dolph-Chebyshev** window (use `scipy.signal.chebwin`) and compare its sidelobe level and beamwidth with the Hann window.